# V3-6 — D1: RGB + Frame-Difference Motion

این مدل RGB featureهای V3-4A را reuse می‌کند و branch جدیدی برای اختلاف مطلق فریم‌های متوالی می‌سازد. خروجی نهایی همچنان در سطح MP4 کامل است.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from v3_d1_motion import Config, build_context, cache_preflight, context_report, ensure_motion_cache, train_model, evaluate_best

config = Config()
# ابتدا preflight را بررسی کن. اجرای کامل فقط با True آغاز می‌شود.
RUN_FULL_MOTION_CACHE = False
MAX_NEW_SEQUENCES = None
RUN_TRAINING_AFTER_COMPLETE_CACHE = True
print({'data_root': str(config.data_root), 'motion_version': config.motion_version, 'primary_aggregation': config.primary_aggregation, 'run_full_motion_cache': RUN_FULL_MOTION_CACHE})

In [ ]:
# 1) Freeze the exact V3-4A train rows and full-MP4 validation rows.
context = build_context(config)
print(context_report(context))
display(context['train'].groupby(['training_role', 'video_label']).agg(windows=('sequence_id', 'size'), videos=('video_id', 'nunique'), loss_weight=('loss_weight', 'first')))
assert context['train'].split.eq('train').all()
assert context['validation'].split.eq('validation').all()


In [ ]:
# 2) Decode two sequences, construct absolute RGB frame differences, and verify [16, 512] motion features.
preflight = cache_preflight(context)
print(preflight)


In [ ]:
# 3) Resumable motion cache. RGB features are already reused from A2-MP-HN1.
motion_cache = ensure_motion_cache(context, run_full_cache=RUN_FULL_MOTION_CACHE, max_sequences=MAX_NEW_SEQUENCES)
print({key: value for key, value in motion_cache.items() if key != 'features_by_sequence'})
if not motion_cache['complete']:
    print('Motion cache is incomplete. Set RUN_FULL_MOTION_CACHE=True only when no other heavy extraction is active.')


In [ ]:
# 4) Train only the RGB-motion fusion head after the cache is complete.
training_result = None
if motion_cache['complete'] and RUN_TRAINING_AFTER_COMPLETE_CACHE:
    training_result = train_model(context, motion_cache)
    print(training_result)
else:
    print('Training waits for the complete motion cache.')


In [ ]:
# 5) Full-MP4 validation and fixed aggregation ablation.
evaluation_result = None
if motion_cache['complete'] and config.model_path.is_file():
    evaluation_result = evaluate_best(context, motion_cache)
    print(evaluation_result['summary']['primary_metrics_validation_selected_threshold'])
    display(evaluation_result['aggregation_ablation'])
else:
    print('Evaluation waits for the trained D1 checkpoint.')
